# Data Preprocessing of DEID_RX_ORDER.csv

In [ ]:
"""
Created on Thursday Nov 13 2025

@author: Laura Rueda
"""

import pandas as pd
import os

## 1. Cleaning

#### 1.1 Import file and correct format (")

In [ ]:
input_file = 'data/deid_rx_order.csv'
temp_file = 'data/deid_rx_order_temp_format.csv'  # Temporal file for format correction
output_file = 'data/output/deid_rx_order_final.csv'      # Final clean file

print(f"Correcting format of the giant file...")

# Correct format line by line (Without using RAM) ---
# We read a line, fix it, and save it to disk immediately.
with open(input_file, 'r', encoding='utf-8') as f_in, \
     open(temp_file, 'w', encoding='utf-8') as f_out:
    
    for i, line in enumerate(f_in):
        line = line.strip()
        if line.startswith('"') and line.endswith('"'):
            line = line[1:-1].replace('""', '"')
        
        f_out.write(line + '\n')
        
        # Print progress every  million rows to know it hasn't frozen
        if i % 1000000 == 0:
            print(f"   Processed {i/1000000:.1f} millions...")

print("Format corrected!")    

Correcting format of the giant file...
   Processed 0.0 millions...
   Processed 1.0 millions...
   Processed 2.0 millions...
   Processed 3.0 millions...
   Processed 4.0 millions...
   Processed 5.0 millions...
   Processed 6.0 millions...
   Processed 7.0 millions...
   Processed 8.0 millions...
   Processed 9.0 millions...
   Processed 10.0 millions...
   Processed 11.0 millions...
   Processed 12.0 millions...
   Processed 13.0 millions...
   Processed 14.0 millions...
   Processed 15.0 millions...
   Processed 16.0 millions...
   Processed 17.0 millions...
   Processed 18.0 millions...
   Processed 19.0 millions...
   Processed 20.0 millions...
Format corrected!


#### 1.2 Load csv into dataframe

In [5]:
print("Loading data into Pandas...")
df = pd.read_csv(temp_file, low_memory=False)    
print(f"   File loaded. Actual dimensions: {df.shape}")

Loading data into Pandas...
   File loaded. Actual dimensions: (20436701, 10)


#### 1.3 Cleaning Process
- Erase columns that are almost empty (RX_EPIC_NAME)
- Fill missing values with Null in ROUTE, DOSE and FREQUENCY
- Sort data chronologically and by patient

In [6]:
# 1. Column cleaning
if 'RX_EPIC_NAME' in df.columns:
    df.drop(columns=['RX_EPIC_NAME'], inplace=True)

# 2. Fill missing values
cols_to_fill = ['ROUTE', 'DOSE', 'FREQUENCY']
missing_before = {c: int(df[c].isna().sum()) for c in cols_to_fill if c in df.columns}
print("Missing values after dropping RX_EPIC_NAME (per column):", missing_before)

for c in cols_to_fill:
    if c in df.columns:
        df[c].fillna('Unknown', inplace=True)

total_filled = sum(missing_before.values())
print(f"Total values filled across {cols_to_fill}: {total_filled}")

# 3. Sorting data chronologically and by patient
if 'Shifted_date' in df.columns:
    df['Shifted_date'] = pd.to_datetime(df['Shifted_date'])
    df.sort_values(by=['PATIENT_ID', 'Shifted_date'], inplace=True)

Missing values after dropping RX_EPIC_NAME (per column): {'ROUTE': 1412639, 'DOSE': 4318435, 'FREQUENCY': 9907033}


C:\Users\rueda\AppData\Local\Temp\ipykernel_27252\1038518651.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[c].fillna('Unknown', inplace=True)


Total values filled across ['ROUTE', 'DOSE', 'FREQUENCY']: 15638107


#### 1.4 Save clean csv

In [7]:
print(f"Saving final file: {output_file}")
df.to_csv(output_file, index=False)

# Delete the temporary file to save disk space
os.remove(temp_file)
print("PROCESS COMPLETED SUCCESSFULLY!")


Saving final file: data/deid_rx_order_final.csv
PROCESS COMPLETED SUCCESSFULLY!


duckdb is a python library that doesn't need server installation and works inside the notebook but with the power of sql: "In-process OLAP database".


In [ ]:
# --- ALTERNATIVE EFFICIENT METHOD USING DUCKDB ---
print("Using SQL (DuckDB) to clean, order and save the data...")

# Connect to the temporary database
con = duckdb.connect()
print("   DuckDB connected.")

# This query does ALL the work of Pandas but without using RAM:
# 1. Reads the temporary file
# 2. Fills nulls (COALESCE)
# 3. Orders by PATIENT_ID and Shifted_date
# 4. Saves the final result directly to disk
query = f"""
COPY (
    SELECT 
        AIM_GROUP, PATIENT_ID, ENCOUNTER_ID, RX_CODE, RX_NAME,
        COALESCE(ROUTE, 'Unknown') AS ROUTE,
        COALESCE(DOSE, 'Unknown') AS DOSE,
        COALESCE(FREQUENCY, 'Unknown') AS FREQUENCY,
        CAST(Shifted_date AS TIMESTAMP) AS Shifted_date
    FROM '{input_file}'
    ORDER BY PATIENT_ID, Shifted_date
) TO '{output_file}' (HEADER, DELIMITER ',')
"""

con.execute(query)
print(f"¡Ready! Clean file saved at: {output_file}")

# --- IMPORTANT (Top 50) ---
# Since DuckDB saves directly to disk, the variable 'df' does not exist in memory.
# We load only the DOSE column so that your next Top 50 cell works:
print("Loading DOSE column for analysis...")
df = pd.read_csv(output_file, usecols=['DOSE'])

## 2. One-Hot-Encoding of DOSE column

In [9]:
# 1. Calculate how many times each dosis appears
# value_counts() counts and orders from most to least frequent
conteo_dosis = df['DOSE'].value_counts()
print(conteo_dosis)

# 2. Show only the TOP 50
print("--- TOP 50 MOST FREQUENT DOSES ---")
print(conteo_dosis.head(50))

# (Optional) If you want to see what percentage of your data these 50 doses represent:
total_rows = len(df)
rows_top_50 = conteo_dosis.head(50).sum()
print(f"\nThese 50 doses cover {rows_top_50} rows.")
print(f"They represent {(rows_top_50 / total_rows) * 100:.2f}% of all your data.")

DOSE
Unknown        4318435
100 mg         1000647
10 mg           760510
20 mg           624914
500 mg          542370
                ...   
0.6083 mg            1
43,820 mcg           1
1,311.72 mg          1
2,513.4 mcg          1
127.6125 mg          1
Name: count, Length: 32426, dtype: int64
--- TOP 50 MOST FREQUENT DOSES ---
DOSE
Unknown        4318435
100 mg         1000647
10 mg           760510
20 mg           624914
500 mg          542370
4 mg            475251
5,000 units     457771
1 Gtt           457050
40 mg           435822
5 mg            406221
50 mg           404820
30 mg           395775
25 mg           369260
1 mg            305434
600 mg          302000
2 mg            284380
1 appl          280456
1,000 mg        280357
300 mg          269288
325 mg          239174
2 Gm            227953
200 mg          212156
5,000 Units     173961
2.5 mg          172305
150 mg          171865
400 mg          165796
15 mg           151788
80 mg           150707
650 mg          1